# 1. Setup and Libraries

In [ ]:
import os
import glob
import json
import numpy as np
import cv2
import mediapipe as mp
import pandas as pd
import concurrent.futures
from scipy.spatial.transform import Rotation
from pathlib import Path
from tqdm import tqdm
import time

# 2. Dataset Directories

In [2]:
dataset_Folder = "G:/My Drive/GP Datasets/fit3d_data/"
file_path = dataset_Folder + 'fit3d_info.json'

# 3. Load FIT3D Metadata

In [3]:
info_JSON = pd.read_json(file_path, typ='series')
print("JSON loaded successfully as a series:")
print(info_JSON.head())

val_subj_names = ['s03', 's11']
all_train = [sub for sub in info_JSON['train_subj_names'] if sub not in val_subj_names]
all_camera_names = info_JSON['all_camera_names']

# Extract all actions from all subjects and find unique values
all_actions_list = [action for actions in info_JSON['subj_to_act'].values() for action in actions]
unique_actions = sorted(list(set(all_actions_list)))

print(f"Number of unique actions: {len(unique_actions)}")
print("Unique actions:")
print(unique_actions)

JSON loaded successfully as a series:
subj_to_act         {'s03': ['band_pull_apart', 'dumbbell_high_pul...
test_subj_names                                       [s02, s12, s13]
train_subj_names             [s03, s04, s05, s07, s08, s09, s10, s11]
all_camera_names             [50591643, 58860488, 60457274, 65906101]
dtype: object
Number of unique actions: 47
Unique actions:
['band_pull_apart', 'barbell_dead_row', 'barbell_row', 'barbell_shrug', 'burpees', 'clean_and_press', 'deadlift', 'diamond_pushup', 'drag_curl', 'dumbbell_biceps_curls', 'dumbbell_curl_trifecta', 'dumbbell_hammer_curls', 'dumbbell_high_pulls', 'dumbbell_overhead_shoulder_press', 'dumbbell_reverse_lunge', 'dumbbell_scaptions', 'man_maker', 'mule_kick', 'neutral_overhead_shoulder_press', 'one_arm_row', 'overhead_extension_thruster', 'overhead_trap_raises', 'pushup', 'side_lateral_raise', 'squat', 'standing_ab_twists', 'w_raise', 'walk_the_box', 'warmup_1', 'warmup_10', 'warmup_11', 'warmup_12', 'warmup_13', 'warmup_14

# 4. Exercise Availability Table

In [4]:
# Create a binary matrix: subjects as columns, exercises as rows
subj_to_act = info_JSON['subj_to_act']
subjects = sorted(subj_to_act.keys())

# Initialize data for the table
table_data = {}
for subj in subjects:
    subj_actions = set(subj_to_act[subj])
    # Check if each unique action is performed by this subject
    table_data[subj] = [action in subj_actions for action in unique_actions]

# Create DataFrame
exercise_table = pd.DataFrame(table_data, index=unique_actions)

# Display the table
print("Exercise Availability Table (True if subject has the exercise):")
display(exercise_table)

Exercise Availability Table (True if subject has the exercise):


,s02,s03,s04,s05,s07,s08,s09,s10,s11,s12,s13
band_pull_apart,True,True,True,True,True,True,True,True,True,True,True
barbell_dead_row,True,True,True,True,True,True,True,True,True,True,True
barbell_row,True,True,True,True,True,True,True,True,True,True,True
barbell_shrug,True,True,True,True,True,True,True,True,True,True,True
burpees,True,True,True,True,True,True,True,True,True,True,True
clean_and_press,True,True,True,True,True,True,True,True,True,True,True
deadlift,True,True,True,True,True,True,True,True,True,True,True
diamond_pushup,True,True,True,True,True,True,True,True,True,True,True
drag_curl,True,True,True,True,True,True,True,True,True,True,True
dumbbell_biceps_curls,True,True,True,True,True,True,True,True,True,True,True


# 5. Mediapipe 3D Pose Evaluation
## 5.1 Configuration & Indices

In [5]:
# MediaPipe Pose setup
mp_pose = mp.solutions.pose

# Joint subset ordered by H36M joint index.
# Joint:        [ L_Hip, L_Knee, L_Ankle,  R_Hip, R_Knee, R_Ankle,  L_Shoulder, L_Elbow, L_Wrist, R_Shoulder, R_Elbow, R_Wrist]
fit3d_indices = [    1,      2,      3,     4,      5,       6,            11,     12,     13,          14,      15,       16]
mp_indices    = [   23,     25,      27,    24,     26,      28,           11,      13,      15,         12,      14,      16]

## 5.2 Utilities (Procrustes Alignment & Metric)

In [6]:
def procrustes_alignment_batched(pred, target):
    """
    Batched Procrustes alignment.
    pred, target: (num_frames, num_joints, 3)
    """
    mu_p = np.mean(pred, axis=1, keepdims=True)
    mu_t = np.mean(target, axis=1, keepdims=True)
    
    p_0 = pred - mu_p
    t_0 = target - mu_t
    
    norm_p = np.linalg.norm(p_0, axis=(1, 2), keepdims=True)
    norm_t = np.linalg.norm(t_0, axis=(1, 2), keepdims=True)
    norm_t = np.where(norm_t == 0, 1e-8, norm_t)
    norm_p = np.where(norm_p == 0, 1e-8, norm_p)
    
    p_0 = p_0 / norm_p
    t_0 = t_0 / norm_t
    
    # Kabsch algorithm (batched)
    H = np.matmul(p_0.transpose(0, 2, 1), t_0)
    U, S, Vt = np.linalg.svd(H)
    
    detR = np.linalg.det(np.matmul(Vt.transpose(0, 2, 1), U.transpose(0, 2, 1)))
    V_adj = Vt.transpose(0, 2, 1).copy()
    V_adj[detR < 0, :, 2] *= -1
    R = np.matmul(V_adj, U.transpose(0, 2, 1))
    
    aligned = np.matmul(p_0, R.transpose(0, 2, 1)) * norm_t + mu_t
    return aligned

def calculate_pa_mpjpe(pred_3d, gt_3d, mp_idx, gt_idx):
    """
    PA-MPJPE (Procrustes-Aligned MPJPE) — the primary evaluation metric.
    Removes the effect of global rotation, translation, and scale.
    pred_3d, gt_3d: full joint arrays in mm
    """
    num_frames = min(len(pred_3d), len(gt_3d))
    if num_frames == 0: return 0.0
    
    p_subset = np.array(pred_3d[:num_frames])[:, mp_idx]
    g_subset = np.array(gt_3d[:num_frames])[:, gt_idx]
    
    p_aligned = procrustes_alignment_batched(p_subset, g_subset)
    dist = np.linalg.norm(p_aligned - g_subset, axis=-1)
    return np.mean(dist)


# 6. Define MediaPipe 3D Processing Function

# 7. Validation Evaluation Pipeline

## 7.1 Generate Validation Tasks

In [8]:
def get_validation_tasks(dataset_base, subj_names, all_camera_names):
    tasks = []
    base_path = Path(dataset_base)
    for subj in subj_names:
        ref_dir = base_path / "train" / subj / "joints3d_25"
        if not ref_dir.exists(): continue
        for ref_file in ref_dir.glob("*.json"):
            action = ref_file.stem
            gt_3d = None
            for cam_id in np.random.choice(all_camera_names, 1, replace=False):
                v_path = base_path / "train" / subj / "videos" / str(cam_id) / f"{action}.mp4"
                if v_path.exists():
                    if gt_3d is None:
                        with open(ref_file, 'r') as f:
                            gt_data = json.load(f)
                            gt_3d = np.array(gt_data.get('joints3d_25', list(gt_data.values())[0])) if isinstance(gt_data, dict) else np.array(gt_data)
                    tasks.append({'subj': subj, 'action': action, 'cam_id': cam_id, 'video_path': str(v_path), 'gt_3d': gt_3d})
    return tasks

## 7.2 Validation Execution Pipeline

In [9]:
def evaluate_validation_set(tasks, predict_func, complexity, mp_indices, fit3d_indices):
    print(f"Starting parallel validation evaluation of {len(tasks)} videos...", flush=True)
    all_pa_mpjpe = []
    all_fps = []
    action_pa_mpjpe = {}
    
    def process_task(t):
        result = predict_func(t['video_path'], complexity)
        if result is None or result[0] is None: return None
        pred_3d_raw, average_fps = result
        
        gt_3d_mm   = t['gt_3d'] * 1000
        pred_3d_mm = pred_3d_raw * 1000.0
        try:
            pa = calculate_pa_mpjpe(pred_3d_mm, gt_3d_mm, mp_indices, fit3d_indices)
            return (t['subj'], t['action'], t['cam_id'], pa, average_fps)
        except Exception as e:
            return ('ERROR', t['subj'], t['action'], t.get('cam_id', 'TestCam'), e, 0.0)
        
    with concurrent.futures.ThreadPoolExecutor(max_workers=3) as executor:
        futures = [executor.submit(process_task, t) for t in tasks]
        for future in tqdm(concurrent.futures.as_completed(futures), total=len(tasks), desc="Evaluating Pose"):
            res = future.result()
            if isinstance(res, tuple) and res[0] == 'ERROR':
                print(f"\nERROR processing {res[1]} | {res[2]} (Cam {res[3]}): {res[4]}")
            else:
                subj, action, cam_id, pa, average_fps = res
                all_pa_mpjpe.append(pa)
                action_pa_mpjpe.setdefault(action, []).append(pa)
                if average_fps > 0:
                    all_fps.append(average_fps)
                
    if all_pa_mpjpe:
        print("\n\n--- PA-MPJPE per Exercise ---")
        for act in sorted(action_pa_mpjpe.keys()):
            print(f"{act:15s}: PA-MPJPE = {np.mean(action_pa_mpjpe[act]):6.2f} mm")
        print(f"\nOverall Mean PA-MPJPE: {np.mean(all_pa_mpjpe):.2f} mm")
        if all_fps:
            print(f"Overall Mean FPS: {np.mean(all_fps):.2f} FPS")
    return all_pa_mpjpe


# 8. Define MediaPipe 2D Processing Function

In [10]:
def predict_mediapipe_2d(video_path, complexity):
    pred_2d_list = []
    cap = cv2.VideoCapture(video_path)
    import time
    total_inference_time = 0.0
    total_frames = 0
    
    mp_pose = mp.solutions.pose
    with mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5, model_complexity=complexity, static_image_mode=False) as pose:
        last_pose = np.zeros((33, 3))
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret: break
                
            image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            image.flags.writeable = False
            
            # 1. Start the high-precision timer
            start_time = time.perf_counter()

            # 2. Run ONLY the model inference
            results = pose.process(image)

            # 3. Stop the timer
            end_time = time.perf_counter()

            # 4. Calculate exactly how long this one frame took
            frame_time = end_time - start_time
            total_inference_time += frame_time
            total_frames += 1
            height, width, _ = frame.shape
            
            if results.pose_landmarks:
                # 2D relative landmarks scaled to image dimensions
                lms = np.array([[lm.x * width, lm.y * height, lm.z * width] for lm in results.pose_landmarks.landmark])
                pred_2d_list.append(lms)
                last_pose = lms
            else:
                pred_2d_list.append(last_pose)
    
    cap.release()
    average_fps = total_frames / total_inference_time if total_inference_time > 0 else 0.0
    if len(pred_2d_list) == 0:
        return None, average_fps
    return np.array(pred_2d_list), average_fps

# 9. Execute Experiments (PA-MPJPE)

In [11]:
val_tasks = get_validation_tasks(dataset_Folder, val_subj_names, all_camera_names)

## Phase 1: 2D vs 3D (complexity = 1)

### Experiment 1: Medipipe 2D – complexity = 1

In [12]:
print("========== Experiment 1: Medipipe 2D – complexity = 1 ==========")
_ = evaluate_validation_set(val_tasks, predict_mediapipe_2d, complexity=1, mp_indices=mp_indices, fit3d_indices=fit3d_indices)

========== Experiment 1: Medipipe 2D – complexity = 1 ==========
Starting parallel validation evaluation of 94 videos...


Evaluating Pose: 100%|██████████| 94/94 [36:02<00:00, 23.01s/it]



--- PA-MPJPE per Exercise ---
band_pull_apart: PA-MPJPE = 175.59 mm
barbell_dead_row: PA-MPJPE = 205.14 mm
barbell_row    : PA-MPJPE = 188.59 mm
barbell_shrug  : PA-MPJPE = 154.17 mm
burpees        : PA-MPJPE = 197.68 mm
clean_and_press: PA-MPJPE = 207.34 mm
deadlift       : PA-MPJPE = 185.16 mm
diamond_pushup : PA-MPJPE = 181.91 mm
drag_curl      : PA-MPJPE = 196.63 mm
dumbbell_biceps_curls: PA-MPJPE = 186.61 mm
dumbbell_curl_trifecta: PA-MPJPE = 174.66 mm
dumbbell_hammer_curls: PA-MPJPE = 192.32 mm
dumbbell_high_pulls: PA-MPJPE = 202.17 mm
dumbbell_overhead_shoulder_press: PA-MPJPE = 171.82 mm
dumbbell_reverse_lunge: PA-MPJPE = 186.43 mm
dumbbell_scaptions: PA-MPJPE = 170.60 mm
man_maker      : PA-MPJPE = 208.27 mm
mule_kick      : PA-MPJPE = 240.40 mm
neutral_overhead_shoulder_press: PA-MPJPE = 179.92 mm
one_arm_row    : PA-MPJPE = 217.46 mm
overhead_extension_thruster: PA-MPJPE = 189.63 mm
overhead_trap_raises: PA-MPJPE = 204.00 mm
pushup         : PA-MPJPE = 183.71 mm
side_later

In [ ]:
# 10. PoseFormer 3D Conversion and Evaluation
import sys
sys.path.insert(0, r'd:\GP\Pose\PoseFormer')
from common.model_poseformer import PoseTransformer
import torch

# Load Model (Untrained randomly initialized because no pre-trained weights were provided)
num_frame = 9 # smaller sliding window
pose_model = PoseTransformer(num_frame=num_frame, num_joints=17, in_chans=2, embed_dim_ratio=32, depth=4, num_heads=8, mlp_ratio=2., qkv_bias=True, qk_scale=None,drop_path_rate=0.1)
if torch.cuda.is_available():
    pose_model = pose_model.cuda()
pose_model.eval()

def mediapipe_to_h36m(np_lms):
    # np_lms: (N, 33, 3) where 3 is x, y, z (but we only need x, y for PoseFormer)
    N = np_lms.shape[0]
    h36m = np.zeros((N, 17, 2))
    lms = np_lms[..., :2] # x, y
    
    h36m[:, 1] = lms[:, 24] # R_Hip
    h36m[:, 2] = lms[:, 26] # R_Knee
    h36m[:, 3] = lms[:, 28] # R_Ankle
    h36m[:, 4] = lms[:, 23] # L_Hip
    h36m[:, 5] = lms[:, 25] # L_Knee
    h36m[:, 6] = lms[:, 27] # L_Ankle
    h36m[:, 0] = (lms[:, 23] + lms[:, 24]) / 2.0 # Pelvis
    
    h36m[:, 11] = lms[:, 11] # L_Shoulder
    h36m[:, 12] = lms[:, 13] # L_Elbow
    h36m[:, 13] = lms[:, 15] # L_Wrist
    h36m[:, 14] = lms[:, 12] # R_Shoulder
    h36m[:, 15] = lms[:, 14] # R_Elbow
    h36m[:, 16] = lms[:, 16] # R_Wrist
    
    h36m[:, 8] = (lms[:, 11] + lms[:, 12]) / 2.0 # Neck
    h36m[:, 7] = (h36m[:, 0] + h36m[:, 8]) / 2.0 # Spine
    h36m[:, 9] = lms[:, 0] # Nose
    h36m[:, 10] = lms[:, 0] # Head top approx
    
    return h36m

def predict_poseformer(video_path, complexity):
    # Get 2D predictions
    pred_2d_raw, average_fps = predict_mediapipe_2d(video_path, complexity)
    if pred_2d_raw is None:
        return None, average_fps
    
    # Format to H36M 17 joints 2D
    h36m_2d = mediapipe_to_h36m(pred_2d_raw) # (N, 17, 2)
    
    pad = num_frame // 2
    padded = np.pad(h36m_2d, ((pad, pad), (0,0), (0,0)), mode='edge')
    
    pred_3d_list = []
    with torch.no_grad():
        for i in range(h36m_2d.shape[0]):
            window = padded[i:i+num_frame]
            window_tensor = torch.tensor(window).unsqueeze(0).float()
            if torch.cuda.is_available():
                window_tensor = window_tensor.cuda()
            
            # Predict 3D
            out_3d = pose_model(window_tensor) # (batch, 1, 17, 3)
            pred_3d_list.append(out_3d.squeeze(0).squeeze(0).cpu().numpy())
            
    pred_3d = np.array(pred_3d_list) # (N, 17, 3)
    
    # Return scale factor in mm if possible, PoseFormer usually outputs in mm relative to pelvis,
    # or meters depending on training. Assuming meters for compatibility with evaluate_validation_set which multiplies by 1000.
    # Wait, PoseFormer usually returns mm directly. But the evaluate function expects raw output and multiplies by 1000:
    # gt_3d_mm = t['gt_3d'] * 1000
    # pred_3d_mm = pred_3d_raw * 1000.0 
    # To avoid double scaling, if PoseFormer is in mm, we should divide by 1000 here.
    pred_3d = pred_3d / 1000.0
    return pred_3d, average_fps

poseformer_indices = [4, 5, 6, 1, 2, 3, 11, 12, 13, 14, 15, 16]
print("========== Experiment: PoseFormer (Medipipe 2D -> 3D) ==========")
_ = evaluate_validation_set(val_tasks, predict_poseformer, complexity=1, mp_indices=poseformer_indices, fit3d_indices=fit3d_indices)
